In [5]:
import wandb
import matplotlib.pyplot as plt
import numpy as np

In [6]:
WORKSPACE = "fcc_ml"

training_names = {
    "LogMSE": {
        "Transformer": "E_avail_LogMSE_20260224_062703",
        "OmniLearned_Small_Pretrained": "E_avail_LogMSE_PT_1A_20260225_102544",
        "OmniLearned_Small": "E_avail_LogMSE_1A_20260225_102540",
    },
    "HuberW": {
        "Transformer": "E_avail_HuberEWeighted_20260224_131609",
        "OmniLearned_Small": "E_avail_HuberWeighted_NoLog_Cont1_1A_20260224_235204",
        "OmniLearned_Small_Pretrained": "E_avail_HuberWeighted_PT_NoLog_Cont1_1A_20260224_235125",
    },
}

MODEL_CONFIG = {
    "Transformer": {
        "project": "minerva-models",
        "val_loss_key": "eval_loss",
        "train_loss_key": "train_loss",
    },
    "OmniLearned_Small": {
        "project": "omnithings",
        "val_loss_key": "vall loss",
        "train_loss_key": "train loss",
    },
    "OmniLearned_Small_Pretrained": {
        "project": "omnithings",
        "val_loss_key": "vall loss",
        "train_loss_key": "train loss",
    },
}

In [7]:
api = wandb.Api()

def fetch_loss_history(run_name, project, train_key, val_key):
    """Fetch train and val loss series from a W&B run by name."""
    runs = api.runs(f"{WORKSPACE}/{project}", filters={"display_name": run_name})
    if len(runs) == 0:
        raise ValueError(f"No run found with name '{run_name}' in {WORKSPACE}/{project}")
    run = runs[0]
    print(f"  Found run: {run.name} (id={run.id})")

    history = run.scan_history(keys=[train_key, val_key, "_step"])
    steps, train_vals, val_vals = [], [], []
    for row in history:
        step = row.get("_step")
        train_v = row.get(train_key)
        val_v = row.get(val_key)
        if train_v is not None:
            steps.append(step)
            train_vals.append(train_v)
        if val_v is not None:
            val_vals.append((step, val_v))

    train_steps = np.array(steps)
    train_loss = np.array(train_vals)
    val_steps = np.array([s for s, _ in val_vals]) if val_vals else np.array([])
    val_loss = np.array([v for _, v in val_vals]) if val_vals else np.array([])
    return train_steps, train_loss, val_steps, val_loss

In [ ]:
all_curves = {}
for loss_type, models in training_names.items():
    all_curves[loss_type] = {}
    for model_name, run_name in models.items():
        cfg = MODEL_CONFIG[model_name]
        print(f"Fetching {loss_type} / {model_name}: {run_name}")
        train_steps, train_loss, val_steps, val_loss = fetch_loss_history(
            run_name, cfg["project"], cfg["train_loss_key"], cfg["val_loss_key"]
        )
        all_curves[loss_type][model_name] = {
            "train_steps": train_steps,
            "train_loss": train_loss,
            "val_steps": val_steps,
            "val_loss": val_loss,
        }
        print(f"    train points: {len(train_loss)}, val points: {len(val_loss)}")

Fetching LogMSE / Transformer: E_avail_LogMSE_20260224_062703
  Found run: E_avail_LogMSE_20260224_062703 (id=7xnzpmfv)


wandb: WARNING A graphql request initiated by the public wandb API timed out (timeout=19 sec). Create a new API with an integer timeout larger than 19, e.g., `api = wandb.Api(timeout=29)` to increase the graphql timeout.
wandb: WARNING A graphql request initiated by the public wandb API timed out (timeout=19 sec). Create a new API with an integer timeout larger than 19, e.g., `api = wandb.Api(timeout=29)` to increase the graphql timeout.


    train points: 198, val points: 198
Fetching LogMSE / OmniLearned_Small_Pretrained: E_avail_LogMSE_PT_1A_20260225_102544
  Found run: E_avail_LogMSE_PT_1A_20260225_102544 (id=a1u14yg6)
    train points: 0, val points: 0
Fetching LogMSE / OmniLearned_Small: E_avail_LogMSE_1A_20260225_102540
  Found run: E_avail_LogMSE_1A_20260225_102540 (id=vugb1jri)
    train points: 0, val points: 0
Fetching HuberW / Transformer: E_avail_HuberEWeighted_20260224_131609
  Found run: E_avail_HuberEWeighted_20260224_131609 (id=akkakkp4)
    train points: 118, val points: 118
Fetching HuberW / OmniLearned_Small: E_avail_HuberWeighted_NoLog_Cont1_1A_20260224_235204
  Found run: E_avail_HuberWeighted_NoLog_Cont1_1A_20260224_235204 (id=98bxuy20)


In [ ]:
COLORS = {
    "Transformer": "#1f77b4",
    "OmniLearned_Small": "#ff7f0e",
    "OmniLearned_Small_Pretrained": "#2ca02c",
}

for loss_type in training_names:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Loss Curves — {loss_type}", fontsize=14, fontweight="bold")

    for model_name, curves in all_curves[loss_type].items():
        color = COLORS[model_name]
        label = model_name.replace("_", " ")

        if len(curves["train_loss"]) > 0:
            axes[0].plot(curves["train_steps"], curves["train_loss"],
                         label=label, color=color, alpha=0.7, linewidth=0.8)
        if len(curves["val_loss"]) > 0:
            axes[1].plot(curves["val_steps"], curves["val_loss"],
                         label=label, color=color, marker="o", markersize=3, linewidth=1.2)

    for ax, title in zip(axes, ["Train Loss", "Validation Loss"]):
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
for loss_type in training_names:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Loss Curves (log scale) — {loss_type}", fontsize=14, fontweight="bold")

    for model_name, curves in all_curves[loss_type].items():
        color = COLORS[model_name]
        label = model_name.replace("_", " ")

        if len(curves["train_loss"]) > 0:
            axes[0].plot(curves["train_steps"], curves["train_loss"],
                         label=label, color=color, alpha=0.7, linewidth=0.8)
        if len(curves["val_loss"]) > 0:
            axes[1].plot(curves["val_steps"], curves["val_loss"],
                         label=label, color=color, marker="o", markersize=3, linewidth=1.2)

    for ax, title in zip(axes, ["Train Loss", "Validation Loss"]):
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title(title)
        ax.set_yscale("log")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()